## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Sid Luthra
**Date:** 10/2/26

Q1.1: Regression predicts a continuous numerical value, while classification predicts a category or class label.

Q1.2: A confusion table compares predicted classes to actual classes. It shows how many predictions were correct or incorrect for each class.

Q1.3: SSE, or Sum of Squared Errors, measures the total squared difference between predicted and actual values. A lower SSE generally means the model fits the data better.

Q1.4: Overfitting happens when a model learns the training data too closely and performs poorly on new data. Underfitting happens when a model is too simple to capture important patterns.

Q1.5: Testing different k values on separate data helps determine which value generalizes best to unseen data. This helps reduce overfitting or underfitting.

Q1.6: A class label is simple and easy to interpret but does not show how confident the model is. A probability distribution gives more information about uncertainty, but it is slightly more complex to interpret.

In [1]:
#Q2.1
import pandas as pd

df = pd.read_csv("land_mines.csv")
df.head()

,voltage,height,soil,mine_type
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


### Q2.1 Load the data and perform EDA
The dataset contains three predictor variables, `voltage`, `height`, and `soil`, and the target variable `mine_type`, which contains classes 1 through 5.

In [2]:
print(df.shape)
print(df.info())
print(df.describe())
print(df["mine_type"].value_counts().sort_index())

(338, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

Training observations: 169
Testing observations: 169


### Q2.2 Train/test split
I split the data 50/50 into training and testing sets. I used stratification so that each mine type is represented proportionally in both sets.

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

k_values = range(1, 31)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    accuracies.append(accuracy)

results = pd.DataFrame({
    "k": k_values,
    "accuracy": accuracies
})

results

,k,accuracy
0,1,0.402367
1,2,0.437870
2,3,0.378698
3,4,0.355030
4,5,0.313609
5,6,0.325444
6,7,0.313609
7,8,0.313609
8,9,0.349112
9,10,0.337278


In [5]:
best_row = results.loc[results["accuracy"].idxmax()]

best_k = int(best_row["k"])
best_accuracy = best_row["accuracy"]

print("Best k:", best_k)
print("Best accuracy:", best_accuracy)

Best k: 2
Best accuracy: 0.4378698224852071


### Q2.3 Selecting k
I tested values of k from 1 through 30 and compared their classification accuracy on the test set. The value with the highest accuracy was selected as the optimal k.

In [6]:
best_model = KNeighborsClassifier(n_neighbors=best_k)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

In [7]:
from sklearn.metrics import confusion_matrix, accuracy_score

confusion = pd.crosstab(
    y_test,
    y_pred,
    rownames=["Actual"],
    colnames=["Predicted"]
)

print(confusion)

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)

Predicted   1   2   3  4  5
Actual                     
1          25   0   6  4  1
2           0  32   0  3  0
3          12   2   9  6  4
4          12   5   8  8  0
5          15   2  10  5  0

Accuracy: 0.4378698224852071


In [9]:
correct_by_class = confusion.values.diagonal()
total_by_class = confusion.sum(axis=1).values

class_results = pd.DataFrame({
    "mine_type": confusion.index,
    "correct": correct_by_class,
    "total": total_by_class,
    "accuracy": correct_by_class / total_by_class
})

class_results

,mine_type,correct,total,accuracy
0,1,25,36,0.694444
1,2,32,35,0.914286
2,3,9,33,0.272727
3,4,8,33,0.242424
4,5,0,32,0.000000


### Q2.4 Model performance
The optimal k-NN model has an overall accuracy of about 43.8%. The confusion table shows that performance varies by mine type, and some mine types are confused with other classes more frequently than others.

This means overall accuracy alone does not completely describe the model's performance because the model may perform relatively well for one class while performing poorly for another.

### Q2.5 Using the model in practice
I would not use this model by itself to make a final decision about the type of a land mine because incorrect classifications could have serious safety consequences. Instead, I would use the prediction as supporting information along with additional measurements or expert inspection, especially for mine types that the confusion table shows are frequently misclassified.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 1fcd0cf722a9a8704764a9592597522a7a1322c7
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]